# Week 3 · Day 2 — Player Models
**Prediction Models: Match Winner & Top Player**

Built on the real Day 1 feature files:
- `features_team_v1.csv` (pre-match team form features, one row per team per match)
- `features_player_v1.csv` (pre-match player form features, one row per player per match)

...joined against the raw results files to recover the actual targets (which Day 1's feature files deliberately don't contain, to stay leakage-safe):
- `team_matches_home_away_raw.csv` → actual match result (W/L/D), scores
- `afl_players_round_by_round_stats_raw.csv` → actual per-match fantasy points / stats

These models will later be exposed as callable tools to the chat agent.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, brier_score_loss,
    mean_absolute_error, mean_squared_error
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


## 1. Load & join — Match Winner data

`features_team_v1.csv` has no outcome column by design (it's pre-match). We join it to `team_matches_home_away_raw.csv` on `team_name, opponent, year, round, home_away` (case-normalised, since raw team-name casing is inconsistent in places) to recover the actual `result` / scores.


In [2]:
team_feat = pd.read_csv("features_team_v1.csv")
team_actual = pd.read_csv("team_matches_home_away_raw - team_matches_home_away_raw.csv.csv")

for df in (team_feat, team_actual):
    df["team_name_norm"] = df["team_name"].str.strip().str.lower()
    df["opponent_norm"] = df["opponent"].str.strip().str.lower()

match_df = team_feat.merge(
    team_actual[["team_name_norm","opponent_norm","year","round","home_away",
                 "result","team_score","opponent_score","margin","venue"]],
    on=["team_name_norm","opponent_norm","year","round","home_away"],
    how="left",
)
match_df = match_df.drop_duplicates(subset=["match_key","home_away"]).reset_index(drop=True)

print("Unmatched rows (no actual result found):", match_df["result"].isna().sum(), "/", len(match_df))

# Binary target: 1 = this team won. Drop draws (~0.8% of matches) — undefined for binary win/loss.
match_df = match_df.dropna(subset=["result"])
match_df = match_df[match_df["result"] != "D"].copy()
match_df["team_won"] = (match_df["result"] == "W").astype(int)

print(match_df.shape)
match_df.head()


Unmatched rows (no actual result found): 0 / 15808
(15674, 23)


,match_key,team_name,opponent,home_away,year,round,last5_avg_score,last5_win_rate,last3_win_rate,win_streak,...,cum_wins_before_match,ladder_rank_before_match,team_name_norm,opponent_norm,result,team_score,opponent_score,margin,venue,team_won
0,Collingwood Magpies_Melbourne Demons_1983-03-26,Collingwood Magpies,Melbourne Demons,A,1983,1,NaN,NaN,NaN,0,...,0.0,1.0,collingwood magpies,melbourne demons,W,135,125,10,Melbourne Cricket Ground,1
1,Collingwood Magpies_Melbourne Demons_1983-03-26,Melbourne Demons,Collingwood Magpies,H,1983,1,NaN,NaN,NaN,0,...,0.0,1.0,melbourne demons,collingwood magpies,L,125,135,-10,Melbourne Cricket Ground,0
2,Carlton Blues_Richmond Tigers_1983-03-26,Richmond Tigers,Carlton Blues,A,1983,1,NaN,NaN,NaN,0,...,0.0,1.0,richmond tigers,carlton blues,L,76,136,-60,Princes Park,0
3,Geelong Cats_Western Bulldogs_1983-03-26,Geelong Cats,Western Bulldogs,H,1983,1,NaN,NaN,NaN,0,...,0.0,1.0,geelong cats,western bulldogs,W,106,75,31,Waverley Park,1
4,St Kilda Saints_north melbourne kangaroos_1983...,St Kilda Saints,north melbourne kangaroos,A,1983,1,NaN,NaN,NaN,0,...,0.0,1.0,st kilda saints,north melbourne kangaroos,L,87,100,-13,Aegis Park,0


## 2. Load & join — Top Player data

`features_player_v1.csv` similarly has no actual-performance column. Joined to `afl_players_round_by_round_stats_raw.csv` on `player_id, match_date` to recover `fantasy_points` (the standard AFL "who played well" metric) as the regression target.


In [3]:
player_feat = pd.read_csv("features_player_v1.csv")
player_actual = pd.read_csv("afl_players_round_by_round_stats_raw - afl_players_round_by_round_stats_raw.csv.csv")

player_df = player_feat.merge(
    player_actual[["player_id","match_date","fantasy_points","disposals","goals","round"]],
    on=["player_id","match_date"],
    how="left",
)
player_df = player_df.drop_duplicates(subset=["id"]).reset_index(drop=True)
player_df = player_df.dropna(subset=["fantasy_points"]).copy()

player_df["year"] = pd.to_datetime(player_df["match_date"]).dt.year

print("Unmatched rows:", player_feat.shape[0] - player_df.shape[0])
print(player_df.shape)
player_df.head()


Unmatched rows: 0
(274079, 13)


,id,player_id,match_date,team,opponent,last5_avg_disposals,last5_avg_goals,last5_avg_fantasy,fantasy_points,disposals,goals,round,year
0,572877,45852,1983-03-27,Essendon Bombers,Sydney Swans,NaN,NaN,NaN,52,10.0,2.0,1,1983
1,572878,45852,1983-04-04,Essendon Bombers,St Kilda Saints,10.0,2.0,52.0,47,8.0,2.0,2,1983
2,572879,45852,1983-04-09,Essendon Bombers,Fitzroy Lions,9.0,2.0,49.5,28,NaN,NaN,3,1983
3,594072,45679,1983-04-16,St Kilda Saints,Geelong Cats,NaN,NaN,NaN,34,5.0,2.0,4,1983
4,594073,45679,1983-04-24,St Kilda Saints,Sydney Swans,5.0,2.0,34.0,47,10.0,1.0,5,1983


## Time-based train / hold-out split

2025 is a complete season in both files (432 team-rows, ~9.9k player-rows) → held out entirely as the test set. Nothing after the training cut-off is used to build features, so this is a clean forward-looking evaluation.


In [4]:
HOLD_YEAR = 2025

match_train = match_df[match_df["year"] < HOLD_YEAR].copy()
match_hold  = match_df[match_df["year"] == HOLD_YEAR].copy()

player_train = player_df[player_df["year"] < HOLD_YEAR].copy()
player_hold  = player_df[player_df["year"] == HOLD_YEAR].copy()

print(f"Match train: {match_train.shape}, hold-out ({HOLD_YEAR}): {match_hold.shape}")
print(f"Player train: {player_train.shape}, hold-out ({HOLD_YEAR}): {player_hold.shape}")


Match train: (15244, 23), hold-out (2025): (430, 23)
Player train: (264143, 13), hold-out (2025): (9936, 13)


## Task 1 — Baseline Models

### 1a. Match winner baseline
"Always predict home team win" (literal reading of the brief's baseline) — evaluated on the hold-out season.

### 1b. Top player baseline
"Last week's leader repeats" — each player's *previous* actual match score predicts this match's score. Leakage-safe (only uses the past) and avoids the look-ahead bias a same-season average would introduce.


In [5]:
# --- Match winner baseline ---
baseline_pred = (match_hold["home_away"] == "H").astype(int)
baseline_proba = baseline_pred.astype(float)

baseline_metrics = {
    "accuracy": accuracy_score(match_hold["team_won"], baseline_pred),
    "f1": f1_score(match_hold["team_won"], baseline_pred),
    "roc_auc": roc_auc_score(match_hold["team_won"], baseline_proba),
    "brier_score": brier_score_loss(match_hold["team_won"], baseline_proba),
}
print("Match winner baseline ('home team always wins'):", baseline_metrics)


Match winner baseline ('home team always wins'): {'accuracy': 0.5581395348837209, 'f1': 0.5581395348837209, 'roc_auc': 0.5581395348837209, 'brier_score': 0.4418604651162791}


In [6]:
# --- Top player baseline: last week's actual score repeats ---
player_df_sorted = player_df.sort_values(["player_id","match_date"]).copy()
player_df_sorted["prev_fantasy_points"] = player_df_sorted.groupby("player_id")["fantasy_points"].shift(1)

player_hold_b = player_df_sorted[player_df_sorted["year"] == HOLD_YEAR].dropna(subset=["prev_fantasy_points"])

baseline_mae = mean_absolute_error(player_hold_b["fantasy_points"], player_hold_b["prev_fantasy_points"])
baseline_rmse = rmse(player_hold_b["fantasy_points"], player_hold_b["prev_fantasy_points"])

def topk_hit_rate(df, match_key_col, actual_col, pred_col, k=5):
    hits = []
    for _, grp in df.groupby(match_key_col):
        if len(grp) < 2:
            continue
        actual_top = set(grp.sort_values(actual_col, ascending=False).head(k).index)
        pred_top   = set(grp.sort_values(pred_col, ascending=False).head(k).index)
        hits.append(len(actual_top & pred_top) > 0)
    return float(np.mean(hits)) if hits else float("nan")

player_hold_b["match_group"] = (
    player_hold_b[["team","opponent"]].apply(lambda r: "_".join(sorted([r["team"], r["opponent"]])), axis=1)
    + "_" + player_hold_b["match_date"]
)
baseline_topk = topk_hit_rate(player_hold_b, "match_group", "fantasy_points", "prev_fantasy_points", k=5)

print(f"Top player baseline (last week repeats) — MAE: {baseline_mae:.2f}, RMSE: {baseline_rmse:.2f}, "
      f"Top-5 hit rate: {baseline_topk:.2%}")


Top player baseline (last week repeats) — MAE: 22.75, RMSE: 28.82, Top-5 hit rate: 90.28%


**Baselines to beat:**
- Match winner: accuracy / F1 / ROC AUC / Brier score from "home team always wins".
- Top player: MAE / RMSE / top-5 hit rate from "last week's score repeats".


## Task 2 — Match Winner Model

Pipeline: `ColumnTransformer` (scale numeric form features, one-hot encode `home_away`) → classifier. Two model types: **Logistic Regression** and **Gradient Boosting**.

Note on framing: the feature files are *team-per-row*, not match-per-row (no single row holds both teams' stats). The model therefore predicts **"does this team win"** from its own pre-match form — `h2h_win_rate` and `venue_win_rate` already carry the opponent/venue-specific signal. `home_away` is included explicitly.


In [7]:
match_num_features = ["last5_avg_score","last5_win_rate","last3_win_rate","win_streak",
                       "days_rest","h2h_win_rate","venue_win_rate",
                       "cum_wins_before_match","ladder_rank_before_match"]
match_cat_features = ["home_away"]
match_features = match_num_features + match_cat_features
target = "team_won"

for df in (match_train, match_hold):
    df[match_num_features] = df[match_num_features].fillna(match_train[match_num_features].median())

X_train, y_train = match_train[match_features], match_train[target]
X_hold,  y_hold  = match_hold[match_features],  match_hold[target]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), match_num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), match_cat_features),
])

logreg_pipe = Pipeline([("prep", preprocess),
                         ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
gb_pipe = Pipeline([("prep", preprocess),
                     ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE))])

logreg_pipe.fit(X_train, y_train)
gb_pipe.fit(X_train, y_train)
print("Both match-winner models trained.")


Both match-winner models trained.


In [8]:
def evaluate_classifier(pipe, X, y, name):
    proba = pipe.predict_proba(X)[:, 1]
    pred = pipe.predict(X)
    return {
        "model": name,
        "accuracy": accuracy_score(y, pred),
        "f1": f1_score(y, pred),
        "roc_auc": roc_auc_score(y, proba),
        "brier_score": brier_score_loss(y, proba),
    }

results = [
    {"model": "baseline_home_always_wins", **baseline_metrics},
    evaluate_classifier(logreg_pipe, X_hold, y_hold, "logistic_regression"),
    evaluate_classifier(gb_pipe, X_hold, y_hold, "gradient_boosting"),
]
match_results_df = pd.DataFrame(results).set_index("model")
match_results_df


,accuracy,f1,roc_auc,brier_score
model,,,,
baseline_home_always_wins,0.558140,0.558140,0.558140,0.441860
logistic_regression,0.609302,0.603774,0.666155,0.228414
gradient_boosting,0.623256,0.604878,0.671628,0.227566


**Model choice & justification:**

| Model | Accuracy | F1 | ROC AUC | Brier Score |
|---|---|---|---|---|
| Baseline (home always wins) | 0.558 | 0.558 | 0.558 | 0.442 |
| Logistic Regression | 0.609 | 0.604 | 0.666 | 0.228 |
| Gradient Boosting | 0.623 | 0.605 | 0.672 | 0.228 |

Both trained models clearly beat the baseline — ROC AUC jumps from 0.558 to ~0.67, and 
Brier score nearly halves (0.442 → 0.228), so the models are far better calibrated than 
just assuming the home team always wins.

Gradient Boosting is picked as the **final model**: it edges out Logistic Regression on 
every metric (accuracy 62.3% vs 60.9%, ROC AUC 0.672 vs 0.666), while matching it on 
calibration (same Brier score). The gap isn't huge, which suggests the relationship 
between form/ladder features and match outcome is mostly linear/additive rather than 
heavily non-linear — Logistic Regression stays a strong, more interpretable alternative 
if explainability to the agent's end-user matters more than the small accuracy gain.

In [9]:
final_match_model = gb_pipe if match_results_df.loc["gradient_boosting","roc_auc"] >= match_results_df.loc["logistic_regression","roc_auc"] else logreg_pipe
final_match_model_name = "gradient_boosting" if final_match_model is gb_pipe else "logistic_regression"
print("Final match-winner model:", final_match_model_name)


Final match-winner model: gradient_boosting


## Task 3 — Top Player Model

**Framing:** Regression — predict each player's `fantasy_points` for the match from their pre-match form (`last5_avg_disposals`, `last5_avg_goals`, `last5_avg_fantasy`) plus `team`/`opponent` as matchup context, then rank players within a match to get the top-k list. Chosen over pure learning-to-rank because the raw points prediction is independently useful to the agent (e.g. "how many fantasy points is X expected to score"), and ranking falls out of it by sorting within a match.


In [10]:
player_num_features = ["last5_avg_disposals","last5_avg_goals","last5_avg_fantasy"]
player_cat_features = ["opponent"]
player_features_full = player_num_features + player_cat_features
target_p = "fantasy_points"

_train_medians = player_train[player_num_features].median()
for df in (player_train, player_hold):
    df[player_num_features] = df[player_num_features].fillna(_train_medians)

Xp_train, yp_train = player_train[player_features_full], player_train[target_p]
Xp_hold,  yp_hold  = player_hold[player_features_full],  player_hold[target_p]

preprocess_p = ColumnTransformer([
    ("num", StandardScaler(), player_num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), player_cat_features),
])

player_pipe = Pipeline([("prep", preprocess_p),
                         ("reg", GradientBoostingRegressor(random_state=RANDOM_STATE))])
player_pipe.fit(Xp_train, yp_train)
print("Top-player regression model trained.")


Top-player regression model trained.


In [11]:
pred_scores = player_pipe.predict(Xp_hold)
player_hold_eval = player_hold.copy()
player_hold_eval["predicted_score"] = pred_scores
player_hold_eval["match_group"] = (
    player_hold_eval[["team","opponent"]].apply(lambda r: "_".join(sorted([r["team"], r["opponent"]])), axis=1)
    + "_" + player_hold_eval["match_date"]
)

mae = mean_absolute_error(yp_hold, pred_scores)
rmse_val = rmse(yp_hold, pred_scores)
model_topk = topk_hit_rate(player_hold_eval, "match_group", "fantasy_points", "predicted_score", k=5)

player_results_df = pd.DataFrame([
    {"model": "baseline_last_week_repeats", "mae": baseline_mae, "rmse": baseline_rmse, "top5_hit_rate": baseline_topk},
    {"model": "gradient_boosting_reg", "mae": mae, "rmse": rmse_val, "top5_hit_rate": model_topk},
]).set_index("model")
player_results_df


,mae,rmse,top5_hit_rate
model,,,
baseline_last_week_repeats,22.751322,28.821671,0.902778
gradient_boosting_reg,17.922339,22.555111,0.953704


Compare `player_results_df` — the trained model should beat the baseline on MAE/RMSE and top-5 hit rate. If it's close, that's a real (and worth-reporting) finding: recent form alone already carries most of the signal, and bigger gains would need extra features (role/position, opponent's defensive rating, weather, ground size).


## Task 4 — Feature Importance & Sanity Checks


In [12]:
feature_names_match = (
    match_num_features +
    list(final_match_model.named_steps["prep"].named_transformers_["cat"].get_feature_names_out(match_cat_features))
)
if final_match_model_name == "gradient_boosting":
    importances = final_match_model.named_steps["clf"].feature_importances_
else:
    importances = np.abs(final_match_model.named_steps["clf"].coef_[0])

match_importance_df = (
    pd.DataFrame({"feature": feature_names_match, "importance": importances})
    .sort_values("importance", ascending=False)
)
match_importance_df


,feature,importance
10,home_away_H,0.172303
1,last5_win_rate,0.168956
8,ladder_rank_before_match,0.148769
6,venue_win_rate,0.137703
9,home_away_A,0.100284
0,last5_avg_score,0.094039
7,cum_wins_before_match,0.068202
5,h2h_win_rate,0.062260
3,win_streak,0.025304
4,days_rest,0.014165


In [13]:
feature_names_player = (
    player_num_features +
    list(player_pipe.named_steps["prep"].named_transformers_["cat"].get_feature_names_out(player_cat_features))
)
player_importances = player_pipe.named_steps["reg"].feature_importances_

player_importance_df = (
    pd.DataFrame({"feature": feature_names_player, "importance": player_importances})
    .sort_values("importance", ascending=False)
    .head(15)
)
player_importance_df


,feature,importance
2,last5_avg_fantasy,0.922642
0,last5_avg_disposals,0.061405
1,last5_avg_goals,0.009948
12,opponent_Gold Coast Suns,0.002158
20,opponent_Sydney Swans,0.001973
11,opponent_Geelong Cats,0.000701
9,opponent_Fitzroy Lions,0.000280
17,opponent_Port Adelaide Power,0.000189
15,opponent_Melbourne Demons,0.000164
4,opponent_Brisbane Bears,0.000131


**Sanity check:** for the match model, `ladder_rank_before_match`, `last5_win_rate`/`last3_win_rate`, and `home_away` should dominate — matches football intuition (form, standing, home advantage). For the player model, `last5_avg_fantasy` should dominate (recent output is the strongest predictor of near-future output) — if `opponent` one-hot levels outrank the numeric form features, that's worth a second look (could be overfitting to a handful of opponents with few holdout matches).

### Sniff test — 3 held-out matches


In [14]:
sample_rows = match_hold[match_hold["home_away"] == "H"].sample(3, random_state=RANDOM_STATE)

for _, row in sample_rows.iterrows():
    proba = final_match_model.predict_proba(row[match_features].to_frame().T)[0, 1]
    print(f"{row['team_name']} (ladder {row['ladder_rank_before_match']:.0f}, home) vs "
          f"{row['opponent']} — round {row['round']}, {row['year']}")
    print(f"  Model P({row['team_name']} wins) = {proba:.2f}  |  Actual result = {row['result']}")
    print(f"  Manual read: last5_win_rate={row['last5_win_rate']:.2f}, "
          f"venue_win_rate={row['venue_win_rate']:.2f}, h2h_win_rate={row['h2h_win_rate']:.2f}\n")


Richmond Tigers (ladder 4, home) vs Geelong Cats — round 24, 2025
  Model P(Richmond Tigers wins) = 0.39  |  Actual result = L
  Manual read: last5_win_rate=0.20, venue_win_rate=0.49, h2h_win_rate=0.22

Geelong Cats (ladder 1, home) vs Hawthorn Hawks — round PF, 2025
  Model P(Geelong Cats wins) = 0.76  |  Actual result = W
  Manual read: last5_win_rate=1.00, venue_win_rate=0.61, h2h_win_rate=0.48

Brisbane Lions (ladder 1, home) vs Port Adelaide Power — round 17, 2025
  Model P(Brisbane Lions wins) = 0.68  |  Actual result = W
  Manual read: last5_win_rate=0.60, venue_win_rate=0.61, h2h_win_rate=0.57



**Sniff test — 3 held-out matches:**

1. **Richmond Tigers (home) vs Geelong Cats** — model gave Richmond only 39% to win, 
   consistent with their weak recent form (last5_win_rate 0.20) and poor head-to-head 
   record (0.22). Richmond lost — model agreed with the eyeball read.

2. **Geelong Cats (home, ladder 1) vs Hawthorn Hawks** — model gave Geelong 76%, matching 
   their perfect recent form (last5_win_rate 1.00) and solid venue record (0.61). Geelong 
   won — model agreed.

3. **Brisbane Lions (home, ladder 1) vs Port Adelaide Power** — model gave Brisbane 68%, 
   in line with decent-but-not-dominant form across all three signals (0.57–0.61). 
   Brisbane won — model agreed.

All three predictions line up with what the raw form numbers would suggest by eye, and 
all three matched the actual result — no leakage red flags or unexplained disagreements 
in this sample.

## Task 5 — Package Models as Callable Functions

Saved as pipelines + reference lookup tables, wrapped by `predict.py` (standalone module — see file alongside this notebook) so Day 4's LangChain/LangGraph tools can import it directly.


In [15]:
import os
os.makedirs("models", exist_ok=True)

joblib.dump(final_match_model, "models/match_winner_model.joblib")
joblib.dump(player_pipe, "models/top_player_model.joblib")

# --- Reference data needed at inference time ---
# Latest known form snapshot per team (any opponent) — fallback context
team_ref = (match_df.sort_values("match_key")
            .groupby("team_name_norm")
            .tail(1)
            .set_index("team_name_norm")[match_num_features + ["team_name"]])

# Latest known head-to-head-specific snapshot per (team, opponent) pair — preferred context
team_pair_ref = (match_df.sort_values("match_key")
                  .groupby(["team_name_norm", "opponent_norm"])
                  .tail(1)
                  .set_index(["team_name_norm", "opponent_norm"])[match_num_features])

# Latest known form snapshot per player
player_ref = (player_df.sort_values("match_date")
              .groupby("player_id")
              .tail(1)
              .set_index("player_id")[["team", "opponent"] + player_num_features])

# Date range the models were trained/evaluated on — used by predict.py to validate
# the optional `date` argument to predict_match_winner
match_df["match_date"] = pd.to_datetime(match_df["match_key"].str.extract(r"(\d{4}-\d{2}-\d{2})$")[0])
date_range = pd.DataFrame({
    "min_date": [match_df["match_date"].min()],
    "max_date": [match_df["match_date"].max()],
})

team_ref.to_csv("models/team_reference.csv")
team_pair_ref.to_csv("models/team_pair_reference.csv")
player_ref.to_csv("models/player_reference.csv")
date_range.to_csv("models/date_range.csv", index=False)

print("Saved: models/match_winner_model.joblib, models/top_player_model.joblib, "
      "models/team_reference.csv, models/team_pair_reference.csv, "
      "models/player_reference.csv, models/date_range.csv")

Saved: models/match_winner_model.joblib, models/top_player_model.joblib, models/team_reference.csv, models/team_pair_reference.csv, models/player_reference.csv, models/date_range.csv


### Quick smoke test of `predict.py` functions
(Run after `predict.py` is saved next to this notebook — see the standalone module.)


In [16]:
import sys
sys.path.insert(0, ".")
from predict import predict_match_winner, predict_top_player

sample_teams = team_ref["team_name"].tolist()
print(predict_match_winner(sample_teams[0], sample_teams[1]))
print(predict_top_player(team=sample_teams[0], k=5))

# 1. Unknown team
try:
    predict_match_winner("Not A Real Team", sample_teams[1])
except ValueError as e:
    print("Validation caught:", e)

# 2. Date outside available data range
try:
    predict_match_winner(sample_teams[0], sample_teams[1], date="2050-01-01")
except ValueError as e:
    print("Date range validation caught:", e)

# 3. Invalid date format (not a real date at all)
try:
    predict_match_winner(sample_teams[0], sample_teams[1], date="not-a-date")
except ValueError as e:
    print("Date format validation caught:", e)

# 4. k < 1 for top player ranking
try:
    predict_top_player(team=sample_teams[0], k=0)
except ValueError as e:
    print("k validation caught:", e)

{'winner': 'Adelaide Crows', 'probability': 0.713, 'home_team': 'Adelaide Crows', 'away_team': 'Brisbane Bears'}
[{'player_id': 45549, 'team': 'Adelaide Crows', 'predicted_score': 97.9}, {'player_id': 45089, 'team': 'Adelaide Crows', 'predicted_score': 95.1}, {'player_id': 44888, 'team': 'Adelaide Crows', 'predicted_score': 94.8}, {'player_id': 43700, 'team': 'Adelaide Crows', 'predicted_score': 94.7}, {'player_id': 44135, 'team': 'Adelaide Crows', 'predicted_score': 89.3}]
Validation caught: Unknown team 'Not A Real Team'. Known teams include: ['Adelaide Crows', 'Brisbane Bears', 'Brisbane Lions', 'Carlton Blues', 'Collingwood Magpies', 'Essendon Bombers', 'Fitzroy Lions', 'Fremantle Dockers', 'Geelong Cats', 'Gold Coast Suns'] ...
Date range validation caught: Date '2050-01-01' is outside the available data range (1983-03-26 to 2025-09-27).
Date format validation caught: Invalid date format: 'not-a-date'. Expected ISO format, e.g. 'YYYY-MM-DD'.
k validation caught: k must be >= 1
